# 面试问题：Agent 怎样安全并行调用工具，处理依赖、限流、部分失败和结果合并？

**一句话回答**：只有读写集合无冲突且无数据依赖的调用才能并行；调度器同时受全局、工具、租户并发配额控制。结果以 task ID 确定性归并，deadline 到达后按 required/optional 和 quorum 决定取消、部分返回或失败；共享写使用幂等键与乐观版本，不能让两个 Agent 无锁覆盖状态。

本 Notebook 手写独立性分析、有限 worker 列表调度、quorum、deadline、乐观并发和 backpressure。

In [ ]:
from dataclasses import dataclass  # 导入本单元所需的依赖。
import hashlib, heapq, json, math  # 导入本单元所需的依赖。

SEED117=11701  # 计算并保存当前步骤的中间状态。
assert SEED117==11701  # 用受控断言验证关键不变量。
assert heapq.nsmallest(2,[3,1,2])==[1,2]  # 用受控断言验证关键不变量。
assert hashlib.sha256(b"fanout").hexdigest()!=hashlib.sha256(b"join").hexdigest()  # 用受控断言验证关键不变量。

## 1. 调用声明 read/write set 和结果重要性

Tool schema 之外还需要资源访问集合、预计 duration、required/optional、deadline 和 tenant。两个调用若有 write-write 或 write-read 交叉，必须串行或进入事务协调。模型不能自行宣称“无冲突”，资源键由宿主根据规范化参数解析。

In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Call117:  # 定义承载本节状态与行为的数据结构。
    call_id:str; tool:str; reads:frozenset; writes:frozenset; duration:int; required:bool=True; tenant:str="T1"  # 计算并保存当前步骤的中间状态。
    def __post_init__(self):  # 定义本节可复用的核心函数。
        if not self.call_id or self.duration<=0 or self.reads&self.writes: raise ValueError("call_contract")  # 按当前条件选择后续控制路径。
calls117=[Call117("weather","weather",frozenset({"city:sh"}),frozenset(),2),Call117("news","search",frozenset({"news:sh"}),frozenset(),4),Call117("profile","db",frozenset({"user:u1"}),frozenset(),1,False)]  # 计算并保存当前步骤的中间状态。
assert len({c.call_id for c in calls117})==3  # 用受控断言验证关键不变量。
assert sum(c.required for c in calls117)==2  # 用受控断言验证关键不变量。
try: Call117("x","t",frozenset({"a"}),frozenset({"a"}),1); raise AssertionError("read-write overlap accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="call_contract"  # 捕获预期异常并验证失败分支。

## 2. 冲突图决定哪些调用可并行

两调用冲突条件为 `W₁∩(R₂∪W₂)` 或 `W₂∩(R₁∪W₁)` 非空。全读调用天然可并行，但仍可能触发外部 rate limit。跨租户不能因为资源字符串相同就共享结果，缓存键需带租户与权限版本。

In [ ]:
def conflict117(a,b): return bool(a.writes&(b.reads|b.writes) or b.writes&(a.reads|a.writes))  # 定义本节可复用的核心函数。
writer117=Call117("update","db",frozenset(),frozenset({"user:u1"}),2); reader117=Call117("read","db",frozenset({"user:u1"}),frozenset(),1)  # 计算并保存当前步骤的中间状态。
assert not conflict117(calls117[0],calls117[1])  # 用受控断言验证关键不变量。
assert conflict117(writer117,reader117)  # 用受控断言验证关键不变量。
assert conflict117(writer117,Call117("update2","db",frozenset(),frozenset({"user:u1"}),1))  # 用受控断言验证关键不变量。

## 3. 有限并发下用列表调度估计时间线

把 ready 调用按确定顺序分配到最早空闲 worker，可得到教学版甘特时间线。真实调度还要按 tenant 做 weighted fair queue，并分别限制模型、搜索、数据库等资源，避免一次 fan-out 耗尽连接池。

In [ ]:
def list_schedule117(calls,workers):  # 定义本节可复用的核心函数。
    heap=[(0,i) for i in range(workers)]; timeline=[]  # 计算并保存当前步骤的中间状态。
    for c in sorted(calls,key=lambda x:x.call_id):  # 遍历输入元素以累积或检查结果。
        free,w=heapq.heappop(heap); end=free+c.duration; timeline.append({"id":c.call_id,"worker":w,"start":free,"end":end}); heapq.heappush(heap,(end,w))  # 计算并保存当前步骤的中间状态。
    return timeline,max(x["end"] for x in timeline) if timeline else 0  # 返回当前分支计算出的结果。
timeline117,makespan117=list_schedule117(calls117,2)  # 计算并保存当前步骤的中间状态。
assert makespan117==4  # 用受控断言验证关键不变量。
assert len(timeline117)==3 and {x["worker"] for x in timeline117}<={0,1}  # 用受控断言验证关键不变量。
assert makespan117<sum(c.duration for c in calls117)  # 用受控断言验证关键不变量。

## 4. Join 必须确定性且验证结果 schema

网络完成顺序不稳定，若直接拼接会让 Prompt 和缓存随机变化。先按 call ID 归档，验证 schema、来源与租户，再按计划定义顺序聚合。重复回调以 `(run_id, call_id, attempt)` 去重。

In [ ]:
arrivals117=[("news",{"items":3}),("weather",{"temp":28}),("profile",{"lang":"zh"})]  # 计算并保存当前步骤的中间状态。
def deterministic_join117(arrivals,expected):  # 定义本节可复用的核心函数。
    by={k:v for k,v in arrivals}  # 计算并保存当前步骤的中间状态。
    if len(by)!=len(arrivals): raise ValueError("duplicate_result")  # 按当前条件选择后续控制路径。
    return [{"call_id":k,"result":by[k]} for k in expected if k in by]  # 返回当前分支计算出的结果。
joined117=deterministic_join117(arrivals117,["weather","news","profile"])  # 计算并保存当前步骤的中间状态。
assert [x["call_id"] for x in joined117]==["weather","news","profile"]  # 用受控断言验证关键不变量。
assert joined117[0]["result"]=={"temp":28}  # 用受控断言验证关键不变量。
assert deterministic_join117(list(reversed(arrivals117)),["weather","news","profile"])==joined117  # 用受控断言验证关键不变量。

## 5. 部分失败策略区分 required、optional 与 quorum

Sectioning 通常要求所有关键维度；Voting 可在达到 quorum 后提前结束。optional 超时可带缺失标记返回，required 失败则尝试替代工具或整体失败，不能让生成器把缺失结果猜出来。

In [ ]:
def join_status117(calls,outcomes,quorum=None):  # 定义本节可复用的核心函数。
    success={k for k,v in outcomes.items() if v=="ok"}; required={c.call_id for c in calls if c.required}  # 计算并保存当前步骤的中间状态。
    if required-success: return "failed_required"  # 按当前条件选择后续控制路径。
    if quorum is not None and len(success)<quorum: return "no_quorum"  # 按当前条件选择后续控制路径。
    return "complete" if len(success)==len(calls) else "partial"  # 返回当前分支计算出的结果。
assert join_status117(calls117,{"weather":"ok","news":"ok","profile":"timeout"})=="partial"  # 用受控断言验证关键不变量。
assert join_status117(calls117,{"weather":"ok","news":"timeout","profile":"ok"})=="failed_required"  # 用受控断言验证关键不变量。
assert join_status117(calls117,{"weather":"ok","news":"ok","profile":"timeout"},3)=="no_quorum"  # 用受控断言验证关键不变量。

## 6. Deadline 与取消是结构化结果

每个子调用使用父 deadline 的剩余时间，不能各自获得完整超时。父请求取消后传播 cancellation；已执行副作用不能靠取消回滚。deadline 时保留已完成 required 结果，并为未完成项记录 timeout，而非丢失整条 trace。

In [ ]:
def deadline_results117(timeline,deadline):  # 定义本节可复用的核心函数。
    return {x["id"]:("ok" if x["end"]<=deadline else "timeout") for x in timeline}  # 返回当前分支计算出的结果。
at3_117=deadline_results117(timeline117,3); at4_117=deadline_results117(timeline117,4)  # 计算并保存当前步骤的中间状态。
assert set(at3_117.values())=={"ok","timeout"}  # 用受控断言验证关键不变量。
assert all(v=="ok" for v in at4_117.values())  # 用受控断言验证关键不变量。
assert sum(v=="ok" for v in at3_117.values())==2  # 用受控断言验证关键不变量。

## 7. 共享写用乐观并发或单写者

两 worker 读取版本 1 后同时写，第二个必须 compare-and-swap 失败并重读合并，不能 last-write-wins。更简单的方案是 fan-out 只读，所有写在 join 后由单一提交者执行。跨系统写仍需要 saga/幂等。

In [ ]:
store117={"value":0,"version":1}  # 计算并保存当前步骤的中间状态。
def cas117(expected,new_value):  # 定义本节可复用的核心函数。
    if store117["version"]!=expected: return False,store117["version"]  # 按当前条件选择后续控制路径。
    store117["value"]=new_value; store117["version"]+=1; return True,store117["version"]  # 计算并保存当前步骤的中间状态。
first117=cas117(1,10); second117=cas117(1,20)  # 计算并保存当前步骤的中间状态。
assert first117==(True,2)  # 用受控断言验证关键不变量。
assert second117==(False,2)  # 用受控断言验证关键不变量。
assert store117=={"value":10,"version":2}  # 用受控断言验证关键不变量。

## 8. Backpressure、隔离与可观测性

fan-out 数量、嵌套深度、每工具并发与每租户 token 都有上限；队列满时拒绝/降级而不是继续派生任务。trace 记录 parent/child、排队、执行、attempt、取消和结果摘要，用 critical path 区分慢工具与排队拥塞。

In [ ]:
def admit117(active,limit,fanout,max_fanout): return active<limit and fanout<=max_fanout  # 定义本节可复用的核心函数。
manifest117={"schema":1,"global_concurrency":32,"per_tool":{"search":8,"db":4},"max_fanout":10,"join_order":"plan_id","write_consistency":"cas_or_single_writer","deadline":"parent_propagated"}; digest117=hashlib.sha256(json.dumps(manifest117,sort_keys=True).encode()).hexdigest()  # 计算并保存当前步骤的中间状态。
assert admit117(3,4,5,10) and not admit117(4,4,5,10)  # 用受控断言验证关键不变量。
assert not admit117(1,4,11,10) and manifest117["join_order"]=="plan_id"  # 用受控断言验证关键不变量。
assert len(digest117)==64  # 用受控断言验证关键不变量。

## 面试总结

一套完整回答是：**声明 read/write set → 冲突图 → 有限并发/租户公平 → 结果按 ID 确定性 join → required/optional/quorum → 父 deadline 与取消 → CAS/单写者 → backpressure 与 critical-path trace**。并行优化的是墙钟时间，不会自动降低总成本或解决共享状态。

延伸阅读：[Anthropic Parallelization Pattern](https://www.anthropic.com/engineering/building-effective-agents)、[Structured Concurrency](https://vorpus.org/blog/notes-on-structured-concurrency-or-go-statement-considered-harmful/)、[OpenTelemetry Trace 规范](https://opentelemetry.io/docs/specs/otel/trace/)。